# Google Colab edition

Run the setup cell first, then work from top to bottom. Keep the course ZIP in My Drive. Use a fresh Colab runtime when switching dependency profiles. **Select a GPU runtime before running this notebook.** 

Original fastbook exercise notebook linked by Part 1. Public MovieLens/MNIST data download automatically. QUICK_RUN uses a subset of the real data and one training epoch per model. Book text and code retain their upstream licenses; see LICENSE_FASTBOOK.

Source: https://github.com/fastai/fastbook/blob/4aefe0a9050d9894255a878c7d72c04458d60de8/clean/08_collab.ipynb

In [ ]:
# Colab setup. Run this cell before the exercise cells.
from pathlib import Path
import os, sys, zipfile

USE_GOOGLE_DRIVE = True  #@param {type:"boolean"}
QUICK_RUN = False  #@param {type:"boolean"}
ARCHIVE_PATH = "/content/drive/MyDrive/HaiLong_Colab_Exercises_2026-10-03.zip"  #@param {type:"string"}

try:
    from google.colab import drive, files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if USE_GOOGLE_DRIVE:
        if not Path('/content/drive/MyDrive').is_dir():
            drive.mount('/content/drive', timeout_ms=300000)
    COURSE_ROOT = Path('/content/HaiLong_Colab_Exercises')
    if not (COURSE_ROOT / 'colab_runtime.py').is_file():
        archive = Path(ARCHIVE_PATH)
        if not archive.is_file():
            print('Select HaiLong_Colab_Exercises_2026-10-03.zip from your computer.')
            uploaded = files.upload()
            choices = [Path(n) for n in uploaded if n.endswith('.zip')]
            if len(choices) != 1:
                raise ValueError('Upload exactly one course ZIP file.')
            archive = choices[0]
        with zipfile.ZipFile(archive) as z:
            base = Path('/content').resolve()
            for member in z.infolist():
                target = (base / member.filename).resolve()
                if not target.is_relative_to(base) or not member.filename.startswith('HaiLong_Colab_Exercises/'):
                    raise ValueError('This is not the expected course archive.')
            z.extractall(base)
    if not (COURSE_ROOT / 'colab_runtime.py').is_file():
        raise FileNotFoundError('The ZIP must contain HaiLong_Colab_Exercises/colab_runtime.py.')
else:
    # Also permits local checks without assuming the current notebook directory.
    COURSE_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                        if (p / 'colab_runtime.py').is_file()), None)
    if COURSE_ROOT is None:
        raise RuntimeError('Open this notebook in Colab or run it inside the extracted course folder.')

os.environ['COURSE_SMOKE'] = '1' if QUICK_RUN else '0'
os.environ['WANDB_MODE'] = 'offline'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
sys.path.insert(0, str(COURSE_ROOT))
from colab_runtime import setup_course, finish_run
# Filled individually for each notebook by the builder.

COLAB_CONTEXT = setup_course(COURSE_ROOT, '03_fastai', 'book', '15_collaborative_filtering.ipynb', USE_GOOGLE_DRIVE, install=IN_COLAB)

## Full exercise evidence

- Run all exercise cells top to bottom in hosted Colab and preserve their outputs, errors and execution counts.
- Keep QUICK_RUN=False; shortened checks or synthetic replacements for real datasets do not count as full original exercises.
- Preserve plots, models, metrics and run configurations in Drive; a final-cell marker alone is insufficient.

- Run all ML-25M ratings through the five-epoch regression, ten-class cross-entropy and genre-feature models on the same recorded split; preserve common metrics, histories, models, class confusion matrices and benchmark-protocol differences.

- Preserve executable questionnaire crosstab and pure-Python dot-product results.

- Worked objective answers are present as labeled learning notes; review them alongside saved coding outputs. Retain empty learner-only observations/personal reflection until supplied by the learner. Prepared prose is not hosted execution or independent recall evidence.

This cell describes evidence to collect; it does not certify completion. GPU batch caps preserve dataset coverage and epoch counts, but are documented configuration changes.


In [ ]:
import fastbook  # Installed by the Colab setup cell.

In [ ]:
#hide
from fastbook import *

# Collaborative Filtering Deep Dive

## A First Look at the Data

In [ ]:
from fastai.collab import *
from fastai.tabular.all import *
path = untar_data(URLs.ML_100k)

In [ ]:
ratings = pd.read_csv(path/'u.data', delimiter='\t', header=None,
                      names=['user','movie','rating','timestamp'])
ratings.head()

In [ ]:
last_skywalker = np.array([0.98,0.9,-0.9])

In [ ]:
user1 = np.array([0.9,0.8,-0.6])

In [ ]:
(user1*last_skywalker).sum()

In [ ]:
casablanca = np.array([-0.99,-0.3,0.8])

In [ ]:
(user1*casablanca).sum()

## Learning the Latent Factors

## Creating the DataLoaders

In [ ]:
movies = pd.read_csv(path/'u.item',  delimiter='|', encoding='latin-1',
                     usecols=(0,1), names=('movie','title'), header=None)
movies.head()

In [ ]:
ratings = ratings.merge(movies)
ratings.head()
if QUICK_RUN: ratings = ratings.sample(n=min(5000,len(ratings)), random_state=42).reset_index(drop=True)

In [ ]:
dls = CollabDataLoaders.from_df(ratings, item_name='title', bs=64, num_workers=0)
dls.show_batch()

In [ ]:
dls.classes

In [ ]:
n_users  = len(dls.classes['user'])
n_movies = len(dls.classes['title'])
n_factors = 5

user_factors = torch.randn(n_users, n_factors)
movie_factors = torch.randn(n_movies, n_factors)

In [ ]:
one_hot_3 = one_hot(3, n_users).float()

In [ ]:
user_factors.t() @ one_hot_3

In [ ]:
user_factors[3]

## Collaborative Filtering from Scratch

In [ ]:
class Example:
    def __init__(self, a): self.a = a
    def say(self,x): return f'Hello {self.a}, {x}.'

In [ ]:
ex = Example('Sylvain')
ex.say('nice to meet you')

In [ ]:
class DotProduct(Module):
    def __init__(self, n_users, n_movies, n_factors):
        self.user_factors = Embedding(n_users, n_factors)
        self.movie_factors = Embedding(n_movies, n_factors)
        
    def forward(self, x):
        users = self.user_factors(x[:,0])
        movies = self.movie_factors(x[:,1])
        return (users * movies).sum(dim=1)

In [ ]:
x,y = dls.one_batch()
x.shape

In [ ]:
model = DotProduct(n_users, n_movies, 50)
learn = Learner(dls, model, loss_func=MSELossFlat())

In [ ]:
learn.fit_one_cycle(1 if QUICK_RUN else 5, 5e-3)

In [ ]:
class DotProduct(Module):
    def __init__(self, n_users, n_movies, n_factors, y_range=(0,5.5)):
        self.user_factors = Embedding(n_users, n_factors)
        self.movie_factors = Embedding(n_movies, n_factors)
        self.y_range = y_range
        
    def forward(self, x):
        users = self.user_factors(x[:,0])
        movies = self.movie_factors(x[:,1])
        return sigmoid_range((users * movies).sum(dim=1), *self.y_range)

In [ ]:
model = DotProduct(n_users, n_movies, 50)
learn = Learner(dls, model, loss_func=MSELossFlat())
learn.fit_one_cycle(1 if QUICK_RUN else 5, 5e-3)

In [ ]:
class DotProductBias(Module):
    def __init__(self, n_users, n_movies, n_factors, y_range=(0,5.5)):
        self.user_factors = Embedding(n_users, n_factors)
        self.user_bias = Embedding(n_users, 1)
        self.movie_factors = Embedding(n_movies, n_factors)
        self.movie_bias = Embedding(n_movies, 1)
        self.y_range = y_range
        
    def forward(self, x):
        users = self.user_factors(x[:,0])
        movies = self.movie_factors(x[:,1])
        res = (users * movies).sum(dim=1, keepdim=True)
        res += self.user_bias(x[:,0]) + self.movie_bias(x[:,1])
        return sigmoid_range(res, *self.y_range)

In [ ]:
model = DotProductBias(n_users, n_movies, 50)
learn = Learner(dls, model, loss_func=MSELossFlat())
learn.fit_one_cycle(1 if QUICK_RUN else 5, 5e-3)

### Weight Decay

In [ ]:
x = np.linspace(-2,2,100)
a_s = [1,2,5,10,50] 
ys = [a * x**2 for a in a_s]
_,ax = plt.subplots(figsize=(8,6))
for a,y in zip(a_s,ys): ax.plot(x,y, label=f'a={a}')
ax.set_ylim([0,5])
ax.legend();

In [ ]:
model = DotProductBias(n_users, n_movies, 50)
learn = Learner(dls, model, loss_func=MSELossFlat())
learn.fit_one_cycle(1 if QUICK_RUN else 5, 5e-3, wd=0.1)

### Creating Our Own Embedding Module

In [ ]:
class T(Module):
    def __init__(self): self.a = torch.ones(3)

L(T().parameters())

In [ ]:
class T(Module):
    def __init__(self): self.a = nn.Parameter(torch.ones(3))

L(T().parameters())

In [ ]:
class T(Module):
    def __init__(self): self.a = nn.Linear(1, 3, bias=False)

t = T()
L(t.parameters())

In [ ]:
type(t.a.weight)

In [ ]:
def create_params(size):
    return nn.Parameter(torch.zeros(*size).normal_(0, 0.01))

In [ ]:
class DotProductBias(Module):
    def __init__(self, n_users, n_movies, n_factors, y_range=(0,5.5)):
        self.user_factors = create_params([n_users, n_factors])
        self.user_bias = create_params([n_users])
        self.movie_factors = create_params([n_movies, n_factors])
        self.movie_bias = create_params([n_movies])
        self.y_range = y_range
        
    def forward(self, x):
        users = self.user_factors[x[:,0]]
        movies = self.movie_factors[x[:,1]]
        res = (users*movies).sum(dim=1)
        res += self.user_bias[x[:,0]] + self.movie_bias[x[:,1]]
        return sigmoid_range(res, *self.y_range)

In [ ]:
model = DotProductBias(n_users, n_movies, 50)
learn = Learner(dls, model, loss_func=MSELossFlat())
learn.fit_one_cycle(1 if QUICK_RUN else 5, 5e-3, wd=0.1)

## Interpreting Embeddings and Biases

In [ ]:
movie_bias = learn.model.movie_bias.squeeze()
idxs = movie_bias.argsort()[:5]
[dls.classes['title'][i] for i in idxs]

In [ ]:
idxs = movie_bias.argsort(descending=True)[:5]
[dls.classes['title'][i] for i in idxs]

In [ ]:
g = ratings.groupby('title')['rating'].count()
top_movies = g.sort_values(ascending=False).index.values[:1000]
top_idxs = tensor([learn.dls.classes['title'].o2i[m] for m in top_movies])
movie_w = learn.model.movie_factors[top_idxs].cpu().detach()
movie_pca = movie_w.pca(3)
fac0,fac1,fac2 = movie_pca.t()
idxs = list(range(50))
X = fac0[idxs]
Y = fac2[idxs]
plt.figure(figsize=(12,12))
plt.scatter(X, Y)
for i, x, y in zip(top_movies[idxs], X, Y):
    plt.text(x,y,i, color=np.random.rand(3)*0.7, fontsize=11)
plt.show()

### Using fastai.collab

In [ ]:
learn = collab_learner(dls, n_factors=50, y_range=(0, 5.5))

In [ ]:
learn.fit_one_cycle(1 if QUICK_RUN else 5, 5e-3, wd=0.1)

In [ ]:
learn.model

In [ ]:
movie_bias = learn.model.i_bias.weight.squeeze()
idxs = movie_bias.argsort(descending=True)[:5]
[dls.classes['title'][i] for i in idxs]

### Embedding Distance

In [ ]:
movie_factors = learn.model.i_weight.weight
idx = dls.classes['title'].o2i['Silence of the Lambs, The (1991)']
distances = nn.CosineSimilarity(dim=1)(movie_factors, movie_factors[idx][None])
idx = distances.argsort(descending=True)[1]
dls.classes['title'][idx]

## Bootstrapping a Collaborative Filtering Model

## Deep Learning for Collaborative Filtering

In [ ]:
embs = get_emb_sz(dls)
embs

In [ ]:
class CollabNN(Module):
    def __init__(self, user_sz, item_sz, y_range=(0,5.5), n_act=100):
        self.user_factors = Embedding(*user_sz)
        self.item_factors = Embedding(*item_sz)
        self.layers = nn.Sequential(
            nn.Linear(user_sz[1]+item_sz[1], n_act),
            nn.ReLU(),
            nn.Linear(n_act, 1))
        self.y_range = y_range
        
    def forward(self, x):
        embs = self.user_factors(x[:,0]),self.item_factors(x[:,1])
        x = self.layers(torch.cat(embs, dim=1))
        return sigmoid_range(x, *self.y_range)

In [ ]:
model = CollabNN(*embs)

In [ ]:
learn = Learner(dls, model, loss_func=MSELossFlat())
learn.fit_one_cycle(1 if QUICK_RUN else 5, 5e-3, wd=0.01)

In [ ]:
learn = collab_learner(dls, use_nn=True, y_range=(0, 5.5), layers=[100,50])
learn.fit_one_cycle(1 if QUICK_RUN else 5, 5e-3, wd=0.1)

In [ ]:
@delegates(TabularModel)
class EmbeddingNN(TabularModel):
    def __init__(self, emb_szs, layers, **kwargs):
        super().__init__(emb_szs, layers=layers, n_cont=0, out_sz=1, **kwargs)

### Sidebar: kwargs and Delegates

### End sidebar

## Conclusion

## Questionnaire

1. What problem does collaborative filtering solve?
1. How does it solve it?
1. Why might a collaborative filtering predictive model fail to be a very useful recommendation system?
1. What does a crosstab representation of collaborative filtering data look like?
1. Write the code to create a crosstab representation of the MovieLens data (you might need to do some web searching!).
1. What is a latent factor? Why is it "latent"?
1. What is a dot product? Calculate a dot product manually using pure Python with lists.
1. What does `pandas.DataFrame.merge` do?
1. What is an embedding matrix?
1. What is the relationship between an embedding and a matrix of one-hot-encoded vectors?
1. Why do we need `Embedding` if we could use one-hot-encoded vectors for the same thing?
1. What does an embedding contain before we start training (assuming we're not using a pretained model)?
1. Create a class (without peeking, if possible!) and use it.
1. What does `x[:,0]` return?
1. Rewrite the `DotProduct` class (without peeking, if possible!) and train a model with it.
1. What is a good loss function to use for MovieLens? Why? 
1. What would happen if we used cross-entropy loss with MovieLens? How would we need to change the model?
1. What is the use of bias in a dot product model?
1. What is another name for weight decay?
1. Write the equation for weight decay (without peeking!).
1. Write the equation for the gradient of weight decay. Why does it help reduce weights?
1. Why does reducing weights lead to better generalization?
1. What does `argsort` do in PyTorch?
1. Does sorting the movie biases give the same result as averaging overall movie ratings by movie? Why/why not?
1. How do you print the names and details of the layers in a model?
1. What is the "bootstrapping problem" in collaborative filtering?
1. How could you deal with the bootstrapping problem for new users? For new movies?
1. How can feedback loops impact collaborative filtering systems?
1. When using a neural network in collaborative filtering, why can we have different numbers of factors for movies and users?
1. Why is there an `nn.Sequential` in the `CollabNN` model?
1. What kind of model should we use if we want to add metadata about users and items, or information such as date and time, to a collaborative filtering model?

### Further Research

1. Take a look at all the differences between the `Embedding` version of `DotProductBias` and the `create_params` version, and try to understand why each of those changes is required. If you're not sure, try reverting each change to see what happens. (NB: even the type of brackets used in `forward` has changed!)
1. Find three other areas where collaborative filtering is being used, and find out what the pros and cons of this approach are in those areas.
1. Complete this notebook using the full MovieLens dataset, and compare your results to online benchmarks. See if you can improve your accuracy. Look on the book's website and the fast.ai forum for ideas. Note that there are more columns in the full dataset—see if you can use those too (the next chapter might give you ideas).
1. Create a model for MovieLens that works with cross-entropy loss, and compare it to the model in this chapter.

### Objective extension: full MovieLens and loss comparison

The original chapter demonstrations retain MovieLens 100K. The extension below trains the chapter's neural collaborative-filtering recipe on **all 25,000,095 ratings** in the stable [GroupLens ML-25M release](https://grouplens.org/datasets/movielens/25m/), with five full epochs per model and one fixed validation split. It compares regression, cross-entropy and a genre-feature improvement attempt. ML-25M half-star scores require **ten classes**, and cross-entropy probabilities are converted to expected ratings for common RMSE/MAE evaluation. Tensor-backed loading keeps the large dataset feasible in Colab RAM.

The objective questionnaire crosstab and pure-Python dot-product tasks are included. The [online LensKit reference](https://codex.lenskit.org/movielens/ML25M/als-biased.html) uses a different [temporal evaluation protocol](https://codex.lenskit.org/movielens/ML25M/splitting.html) and labels itself work in progress; recorded differences provide context and cannot establish a leaderboard result. QUICK_RUN uses the smaller original data and does not complete the full-data extension.


In [ ]:
# Objective questionnaire Q5 and Q7: crosstab and a pure-Python dot product.
import json, time, hashlib, gc
from pathlib import Path
from torch.utils.data import Dataset,DataLoader
from colab_runtime import download,safe_extract
rating_crosstab=pd.crosstab(ratings['user'],ratings['title'],values=ratings['rating'],aggfunc='mean')
display(rating_crosstab.iloc[:8,:8])
Path('results').mkdir(exist_ok=True)
rating_crosstab.iloc[:20,:20].to_csv('results/movielens_100k_crosstab_preview.csv')
left=[.9,.8,-.6];right=[.98,.9,-.9]
python_dot_product=sum(a*b for a,b in zip(left,right))
assert abs(python_dot_product-2.142)<1e-12
print('Pure-Python dot product:',python_dot_product)
del rating_crosstab

# Full-data extension: use every rating in the stable public ML-25M release.
# QUICK_RUN is explicitly a check of the original 100k data, not this homework.
full_path=Path('data/ml-25m')
if QUICK_RUN:
    full_ratings=ratings[['user','movie','rating']].copy()
    full_movies=movies.copy();dataset_label='ML-100k QUICK CHECK'
else:
    if not (full_path/'ratings.csv').is_file():
        archive=download('https://files.grouplens.org/datasets/movielens/ml-25m.zip','data/ml-25m.zip')
        safe_extract(archive,Path('data'))
    full_ratings=pd.read_csv(full_path/'ratings.csv',usecols=['userId','movieId','rating'],
                             dtype={'userId':'int32','movieId':'int32','rating':'float32'}).rename(columns={'userId':'user','movieId':'movie'})
    full_movies=pd.read_csv(full_path/'movies.csv').rename(columns={'movieId':'movie'})
    assert len(full_ratings)==25000095,'All ML-25M ratings are required, without downsampling.'
    dataset_label='ML-25M all 25,000,095 ratings'

# Tensor-backed datasets avoid materializing a 162k-by-62k dense user/movie table.
user_codes,user_ids=pd.factorize(full_ratings['user'],sort=True)
movie_codes,movie_ids=pd.factorize(full_ratings['movie'],sort=True)
full_x=torch.from_numpy(np.column_stack((user_codes,movie_codes))).long()
full_y=torch.from_numpy(full_ratings['rating'].to_numpy(copy=True)).float()
levels=torch.tensor(sorted(full_ratings['rating'].unique()),dtype=torch.float32)
class_y=torch.searchsorted(levels,full_y).long()
assert torch.allclose(levels[class_y],full_y)
assert QUICK_RUN or len(levels)==10 # Half-star ratings 0.5,...,5.0 need ten classes.
permutation=np.random.default_rng(42).permutation(len(full_y))
valid_indices=permutation[:len(permutation)//5];train_indices=permutation[len(permutation)//5:]
split_sha=hashlib.sha256(valid_indices.tobytes()).hexdigest()
np.save('results/movielens_full_valid_indices.npy',valid_indices)
np.savez('results/movielens_full_id_maps.npz',user_ids=np.asarray(user_ids),movie_ids=np.asarray(movie_ids),rating_levels=levels.numpy())
class RatingRows(Dataset):
    def __init__(self,indices,classification=False):self.indices,self.classification=indices,classification
    def __len__(self):return len(self.indices)
    def __getitem__(self,i):
        index=int(self.indices[i]);return full_x[index],(class_y if self.classification else full_y)[index]

FULL_BATCH_SIZE=1024
def full_rating_loaders(classification=False):
    generator=torch.Generator().manual_seed(42)
    train_dl=DataLoader(RatingRows(train_indices,classification),batch_size=FULL_BATCH_SIZE,shuffle=True,num_workers=0,generator=generator)
    valid_dl=DataLoader(RatingRows(valid_indices,classification),batch_size=FULL_BATCH_SIZE,shuffle=False,num_workers=0)
    return DataLoaders(train_dl,valid_dl,device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'))

# Extra movie columns are static genre features; no held-out ratings become inputs.
movie_table=full_movies.set_index('movie').reindex(movie_ids)
genre_frame=movie_table['genres'].fillna('').str.get_dummies('|') if 'genres' in movie_table else pd.DataFrame(index=movie_table.index)
genre_features=torch.tensor(genre_frame.to_numpy(),dtype=torch.float32)
class FullMovieLensNN(nn.Module):
    def __init__(self,classification=False,use_genres=False):
        super().__init__();self.classification=classification;self.use_genres=use_genres
        self.users=Embedding(len(user_ids),50);self.movies=Embedding(len(movie_ids),50)
        self.register_buffer('genre_features',genre_features)
        width=100+(genre_features.shape[1] if use_genres else 0)
        self.layers=nn.Sequential(nn.Linear(width,100),nn.ReLU(),nn.Linear(100,len(levels) if classification else 1))
    def forward(self,x):
        parts=[self.users(x[:,0]),self.movies(x[:,1])]
        if self.use_genres:parts.append(self.genre_features[x[:,1]])
        outputs=self.layers(torch.cat(parts,dim=1))
        return outputs if self.classification else torch.sigmoid(outputs)*5.5

@torch.no_grad()
def full_rating_evaluation(learner,classification):
    learner.model.eval();device=next(learner.model.parameters()).device
    squared=absolute=correct=rows=0;confusion=torch.zeros(len(levels),len(levels),dtype=torch.int64)
    for cats,targets in learner.dls.valid:
        outputs=learner.model(cats.to(device)).cpu()
        if classification:
            probabilities=outputs.softmax(-1);predictions=probabilities@levels
            expected_classes=targets.long().reshape(-1);truth=levels[expected_classes];chosen=outputs.argmax(-1)
        else:
            predictions=outputs.reshape(-1);truth=targets.float().reshape(-1)
            expected_classes=torch.searchsorted(levels,truth)
            chosen=(predictions[:,None]-levels[None,:]).abs().argmin(-1)
        errors=predictions-truth;squared+=float(errors.square().sum());absolute+=float(errors.abs().sum())
        correct+=int((chosen==expected_classes).sum());rows+=len(truth)
        confusion+=torch.bincount(expected_classes*len(levels)+chosen,minlength=len(levels)**2).reshape(len(levels),len(levels))
    assert rows==len(valid_indices)
    return {'rmse':(squared/rows)**.5,'mae':absolute/rows,'rating_class_accuracy':correct/rows,'validation_rows':rows},confusion

experiments=[('nn_mse',False,False),('nn_cross_entropy',True,False),('nn_mse_with_genres',False,True)]
comparison=[];full_epochs=1 if QUICK_RUN else 5
for name,classification,use_genres in experiments:
    set_seed(42);experiment_dls=full_rating_loaders(classification)
    model=FullMovieLensNN(classification,use_genres)
    objective=CrossEntropyLossFlat() if classification else MSELossFlat()
    experiment=Learner(experiment_dls,model,loss_func=objective)
    start=time.perf_counter();experiment.fit_one_cycle(full_epochs,5e-3,wd=.01)
    metrics,confusion=full_rating_evaluation(experiment,classification)
    row={'model':name,'dataset':dataset_label,'source_rows':len(full_y),'training_rows':len(train_indices),
         'validation_split_sha256':split_sha,'epochs':full_epochs,'batch_size':FULL_BATCH_SIZE,
         'class_levels':levels.tolist(),'objective':'cross_entropy' if classification else 'mse',
         'genres_used':bool(use_genres and genre_features.shape[1]),'genre_columns':genre_frame.columns.tolist(),
         'parameters':sum(p.numel() for p in model.parameters()),
         'training_seconds':time.perf_counter()-start,'quick_run':QUICK_RUN,**metrics}
    assert np.isfinite(row['rmse']) and np.isfinite(row['mae'])
    comparison.append(row)
    torch.save({'state_dict':model.state_dict(),'architecture':repr(model),'configuration':row},f'models/{name}_full_movielens.pt')
    pd.DataFrame(confusion.numpy(),index=levels.tolist(),columns=levels.tolist()).to_csv(f'results/{name}_rating_confusion.csv')
    pd.DataFrame(experiment.recorder.values,columns=['train_loss','valid_loss']).to_csv(f'results/{name}_training_history.csv',index=False)
    pd.DataFrame(comparison).to_csv('results/full_movielens_model_comparison.csv',index=False)
    del experiment,model,experiment_dls;gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()

# Online context: same release, different split/model/tuning. No leaderboard claim.
benchmark={'source':'https://codex.lenskit.org/movielens/ML25M/als-biased.html',
 'split_source':'https://codex.lenskit.org/movielens/ML25M/splitting.html',
 'reference_rmse':.7918119847363368,'reference_model':'LensKit ALS biased MF, 471 factors, tuned parameters',
 'reference_protocol':'global temporal train/tune/test windows; test users restricted to those seen in training',
 'our_protocol':'seed42 random row 80/20 validation split, no separate test/tuning window',
 'directly_comparable':False,'source_status':'work in progress; illustrative published online result, not a validated target',
 'differences_from_reference':{r['model']:r['rmse']-.7918119847363368 for r in comparison},
 'genre_improvement_rmse':comparison[0]['rmse']-comparison[2]['rmse'],
 'cross_entropy_minus_mse_rmse':comparison[1]['rmse']-comparison[0]['rmse']}
(Path('results')/'full_movielens_benchmark_context.json').write_text(json.dumps(benchmark,indent=2))
(Path('results')/'full_movielens_run_evidence.json').write_text(json.dumps({'models':comparison,'benchmark':benchmark,
 'dataset_source':'https://grouplens.org/datasets/movielens/25m/',
 'dataset_terms':'https://files.grouplens.org/datasets/movielens/ml-25m-README.html',
 'questionnaire_code_mapping':{'Q5':'crosstab above','Q7':'pure-Python dot product above','Q13':'original Example class','Q15':'original DotProduct training cells'},
 'provider_course_completion_claimed':False},indent=2))
display(pd.DataFrame(comparison));print(json.dumps(benchmark,indent=2))
print('Cross-entropy predicts ten half-star classes on ML-25M; its probability-weighted expected rating allows an RMSE/MAE comparison with regression.')
print('Genre features improve RMSE only if the measured difference above is positive. Online benchmark protocols differ; the difference is descriptive, not a ranking.')

Save your outputs

Run the following cell after finishing. Choose **File > Save** in Colab to preserve your code and displayed results. Files in `results/` and `models/` persist in Drive when enabled.

In [ ]:
finish_run(COLAB_CONTEXT)

## Worked objective answers — preparation notes

These are authored worked answers keyed to the original question numbers above.
They preserve the prompts and teaching code. They do not attest to independent
recall, understanding, personal experience, a hosted run, or course completion.
Code questions point to runnable demonstrations; their execution evidence must
come from saved Colab outputs. Research comparisons remain conditional on those
outputs. No learner reflection or questionnaire form has been filled on your behalf.


### Questionnaire Q1–Q12: representation and prediction

**Q1.** Collaborative filtering predicts a user's preference for an item using
observed user–item interactions, including items that user has not rated.

**Q2.** In this notebook it learns one factor vector per user and movie. Their dot
product, together with biases when included, predicts a rating. Training adjusts
the factors to reduce error on observed ratings; similar interaction patterns can
yield similar representations.

**Q3.** A low rating-prediction error does not establish a useful ranking. A system
also needs suitable candidates, new-user handling, freshness, diversity and an
evaluation that reflects future use. Observed ratings are a selected subset of
items, and recommendations can change what users encounter.

**Q4.** A ratings crosstab has users as rows and movies as columns, with a rating
where observed and a missing value otherwise. Missing means unobserved, not a
zero-star dislike.

**Q5 — executable demonstration.** The added questionnaire code uses
`pd.crosstab(ratings.user, ratings.title, values=ratings.rating, aggfunc='mean')`
on the original MovieLens table and saves a preview. Aggregation defines how any
duplicate pair is handled. [pandas crosstab](https://pandas.pydata.org/docs/reference/api/pandas.crosstab.html).

**Q6.** A latent factor is a learned coordinate used to represent preference or item
properties. It is latent because the dataset does not provide that coordinate as
a labeled attribute. Individual dimensions need not have a stable human meaning.

**Q7 — executable demonstration.** A dot product sums pairwise products:
`sum(a*b for a,b in zip(user_factors, movie_factors))`. For the added equal-length
Python lists it is `0.9*0.98 + 0.8*0.9 + (-0.6)*(-0.9) = 2.142`. This is pure
Python; the cell also checks the result.

**Q8.** `DataFrame.merge` joins tables using specified key columns or indices. Here,
joining on `movie` attaches each movie title to its rating rows. Join type and key
uniqueness matter: duplicate keys on both sides can multiply rows.
[pandas merge](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html).

**Q9.** An embedding matrix has one learnable row per categorical identity and one
column per factor. Looking up a user or movie returns that row.

**Q10.** Multiplying a one-hot row vector by the embedding matrix selects the same
row as an embedding lookup. Multiple one-hot rows select multiple embeddings.

**Q11.** `Embedding` performs that lookup without materializing a large, mostly
zero one-hot matrix. It stores trainable weights and fits into the model and
optimizer machinery.

**Q12.** Without pretrained weights, the rows start at an initialization, usually
random. They have no learned preference meaning until training.
[PyTorch Embedding](https://docs.pytorch.org/docs/stable/generated/torch.nn.Embedding.html).

### Questionnaire Q13–Q25: implementation and regularization

**Q13 — executable demonstration.** The original `Example` class defines
`__init__` and `say`; the following cells instantiate it and call the method. This
supplies a worked class example. Whether you can recreate it without looking is
your own practice, and is not claimed here.

**Q14.** For the two-dimensional input in `DotProduct`, `x[:,0]` selects every row
of column zero: the encoded user IDs. `x[:,1]` selects movie IDs.

**Q15 — executable demonstration.** The original `DotProduct` definitions look up
user/movie rows, multiply them elementwise and sum over factors. The bounded
version adds `sigmoid_range`; the original learner cells train it. Preserve their
outputs to demonstrate training; independent reconstruction without looking is
not asserted.

**Q16.** Mean squared error is a natural loss when the target is a numerical rating
and squared rating error is the objective. It preserves ordering and penalizes
large misses more heavily. This choice does not alone measure recommendation
quality.
[PyTorch MSELoss](https://docs.pytorch.org/docs/stable/generated/torch.nn.MSELoss.html).

**Q17.** Cross-entropy treats each distinct rating value as a class. The network
must return one logit per class, and targets must be integer class indices, not
raw star values. The prepared ML-25M comparison uses all ten half-star levels
from 0.5 to 5.0 without rounding; its expected rating is the probability-weighted
sum of those values. Compare rating RMSE/MAE as well as class accuracy, because
cross-entropy does not encode distance between adjacent rating classes.
[PyTorch CrossEntropyLoss](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html),
[ML-25M rating vocabulary](https://files.grouplens.org/datasets/movielens/ml-25m-README.html).

**Q18.** User bias captures a user's tendency to score high or low; movie bias
captures a movie's overall tendency to receive high or low ratings. The factor
interaction can then model preferences beyond those offsets.

**Q19.** In the basic SGD formulation, weight decay corresponds to L2
regularization. Decoupled weight decay, such as AdamW, is not generally identical
to adding an L2 penalty to an adaptive optimizer's loss.

**Q20.** With the convention `L = L_data + (lambda/2)*sum(w_i**2)`, the penalty
is half lambda times squared weight norm. An SGD step is
`w_new = (1 - learning_rate*lambda)*w - learning_rate*grad(L_data)`.

**Q21.** The derivative of that penalty with respect to each weight is `lambda*w_i`.
Subtracting the resulting gradient shrinks it toward zero, subject to the data
gradient. Omitting the one-half in Q20 gives `2*lambda*w_i` instead.
[PyTorch SGD](https://docs.pytorch.org/docs/stable/generated/torch.optim.SGD.html),
[decoupled weight-decay paper](https://arxiv.org/abs/1711.05101).

**Q22.** Constraining weights can reduce fitting of noise and weakly supported
interactions. It is a regularization tradeoff, not a guarantee: excessive decay
underfits. Select it using held-out performance.

**Q23.** `torch.argsort(tensor)` returns indices that would place values in sorted
order, along the chosen dimension; `descending=True` reverses the order.
[PyTorch argsort](https://docs.pytorch.org/docs/stable/generated/torch.argsort.html).

**Q24.** Learned movie biases generally do not equal mean movie ratings. They are
estimated jointly with user effects and factors and may be regularized. Raw
means do not adjust for which users rated each movie.

**Q25.** `print(model)` shows the module hierarchy. `model.named_modules()` yields
module names and details; `model.named_parameters()` yields trainable parameter
names, shapes and values. The original notebook prints its models.
[PyTorch Module](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html).

### Questionnaire Q26–Q31: a recommendation system

**Q26.** The bootstrapping or cold-start problem is that a new user or item has
little or no interaction history from which to learn a useful factor vector.

**Q27.** For new users, ask for a few preferences, offer broadly useful candidates,
or use consented relevant attributes. For new movies, use genres and other item
features or initial exploration. A hybrid model can combine metadata with
interaction evidence; check its actual new-user/new-item performance.

**Q28.** Displayed recommendations influence subsequent ratings and clicks. If
training treats these observations as neutral, already-exposed items can dominate
and reduce discovery. Log exposure and evaluate coverage, changes over time and
appropriate exploration, as well as rating error.

**Q29.** A neural model concatenates user and movie vectors, so their lengths can
differ. A direct elementwise product or dot product requires matching lengths
unless a projection first maps them into a common space.

**Q30.** `nn.Sequential` chains the hidden layers and output transformation in
`CollabNN`. It passes each layer's output to the next, avoiding a separate manual
forward expression for every layer.

**Q31.** Use a hybrid neural/tabular model that concatenates categorical embeddings
and normalized continuous metadata with user/item factors. Time features need a
validation split that reflects the intended future prediction; otherwise metadata
can leak information unavailable at prediction time.
[fast.ai collaborative-filtering tutorial](https://docs.fast.ai/tutorial.collab.html),
[fast.ai collab API](https://docs.fast.ai/collab.html).

### Further Research FR1–FR4

**FR1 — parameter implementation.** `Embedding` is a module and is called with
indices: `self.user_factors(x[:,0])`. The `create_params` version stores a registered
`nn.Parameter` and indexes its tensor: `self.user_factors[x[:,0]]`. Both register
weights for optimization; an ordinary unregistered tensor would not. Embedding
biases have shape `(n_users,1)` and factor sums use `keepdim=True`; parameter
biases have shape `(n_users,)` and sums omit that dimension. Mixing `(batch,1)`
with `(batch,)` can broadcast to `(batch,batch)`, so those changes belong together.
The original notebook implements both alternatives.

**FR2 — three documented uses.** Product discovery uses item–item interaction
similarity ([Amazon's original research](https://doi.org/10.1109/MIC.2003.1167344));
playlist continuation uses collaborative track/playlist information
([playlist-continuation paper](https://arxiv.org/abs/1808.04288)); news filtering
has used reader ratings ([original GroupLens paper](https://doi.org/10.1145/192844.192905)).
The following tradeoffs are an application analysis, not measured results from
this notebook: product CF can capture cross-category preferences but struggles
with new inventory and exposure bias; music CF can discover related tracks but
may favor popular tracks and miss a listening session's context; news CF can
reduce an overwhelming article stream but suffers from rapid item turnover and
can narrow exposure. All three need cold-start and feedback-loop evaluation.

**FR3 — full-data and benchmark comparison.** The added full-data exercise uses
all 25,000,095 ML-25M ratings, records one fixed split and trains a regression
baseline plus a genre-feature model for five epochs each. It saves shared metrics,
histories and artifacts. Its online LensKit reference uses a different temporal
protocol; the recorded benchmark is a context reference rather than a directly
comparable score or a validated target. Use the saved hosted results to say whether
genres improved this split. No improvement or benchmark reproduction is asserted
before execution.
[ML-25M source](https://grouplens.org/datasets/movielens/25m/),
[LensKit protocol](https://codex.lenskit.org/movielens/ML25M/splitting.html).

**FR4 — cross-entropy comparison.** The added ten-class model uses the same ratings
and split as the MSE model, saves a confusion matrix, and evaluates common RMSE,
MAE and classification metrics. Five training epochs and all ten original rating
levels are retained. Read its saved comparison table to determine which objective
worked better here; the outcome remains unreported until hosted execution.